# 00 Platform Setup Catalog Schema

Bootstrap the Frankfurt ADS-B lakehouse schemas and table contracts for both the original MVP layer and the new PRU-inspired V2 cellized workflow.

This notebook is safe to rerun. It uses `CREATE SCHEMA IF NOT EXISTS` and `CREATE TABLE IF NOT EXISTS` for all managed objects.


# V2 Design Defaults

The V2 research path keeps the original Bronze/V1 Gold tables intact and adds a PRU-inspired terminal adaptation for `EDDF`.

- Horizontal cell size: `10 x 10 NM`
- Vertical cell size: `3000 ft`
- Analysis window: `15 minutes`
- Altitude range: `SFC-FL100` and `FL100-FL245` are retained as project reference bands, while V2 cellization uses configurable `SFC-FL245` voxel bins
- Projection mode: `local_nm`
- Hotspot stability rule: `minimum_active_windows >= 3`
- Legacy baseline components remain documented: `traffic_load`, `interaction`, `flow_structure`


In [ ]:
from pathlib import Path
import yaml

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

repo_root = Path.cwd().resolve().parent
with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)
with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
catalog = default_catalog
if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    catalog = dbutils.widgets.get("catalog").strip() or default_catalog

focus_airport = region_config["focus_airport"]
ingestion_radius_nm = int(region_config["ingestion_radius_nm"])
bbox = region_config["regional_bbox"]
complexity_window_minutes = int(pipeline_config["time_windows"]["complexity_window_minutes"])
trend_window_minutes = int(pipeline_config["time_windows"]["trend_window_minutes"])
complexity_components = pipeline_config["complexity_components"]
altitude_bands = region_config["altitude_bands"]

schemas = ["ref", "brz_adsb", "brz_weather", "slv_airspace", "gld_airspace", "obs"]

v2_defaults = {
    "horizontal_cell_nm": 10,
    "vertical_cell_ft": 3000,
    "analysis_window_minutes": 15,
    "minimum_active_windows": 3,
    "min_altitude_ft": 0,
    "max_altitude_ft": 24500,
    "projection_mode": "local_nm",
}


In [ ]:
table_specs = {
    "ref.project_scope": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.ref.project_scope (
        scope_id STRING NOT NULL,
        focus_airport STRING NOT NULL,
        ingestion_radius_nm INT NOT NULL,
        bbox_min_latitude DOUBLE NOT NULL,
        bbox_max_latitude DOUBLE NOT NULL,
        bbox_min_longitude DOUBLE NOT NULL,
        bbox_max_longitude DOUBLE NOT NULL,
        complexity_window_minutes INT NOT NULL,
        trend_window_minutes INT NOT NULL,
        complexity_components STRING NOT NULL,
        created_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Reference scope configuration for the Frankfurt case study.'
    """,
    "ref.altitude_bands": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.ref.altitude_bands (
        band_id STRING NOT NULL,
        band_label STRING NOT NULL,
        sort_order INT,
        min_altitude_ft DOUBLE NOT NULL,
        max_altitude_ft DOUBLE NOT NULL,
        created_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Reference altitude bands used by the Frankfurt case study.'
    """,
    "ref.grid_cells": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.ref.grid_cells (
        grid_id STRING NOT NULL,
        min_latitude DOUBLE NOT NULL,
        max_latitude DOUBLE NOT NULL,
        min_longitude DOUBLE NOT NULL,
        max_longitude DOUBLE NOT NULL,
        center_latitude DOUBLE,
        center_longitude DOUBLE,
        grid_resolution_deg DOUBLE,
        active_flag BOOLEAN NOT NULL,
        created_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Reference rectangular grid cells used for Frankfurt hotspot analysis.'
    """,
    "ref.cell_schemes_v2": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.ref.cell_schemes_v2 (
        cell_scheme_id STRING NOT NULL,
        focus_airport STRING NOT NULL,
        horizontal_cell_nm DOUBLE NOT NULL,
        vertical_cell_ft DOUBLE NOT NULL,
        analysis_window_minutes INT NOT NULL,
        min_altitude_ft DOUBLE NOT NULL,
        max_altitude_ft DOUBLE NOT NULL,
        projection_mode STRING NOT NULL,
        minimum_active_windows INT NOT NULL,
        created_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Reference V2 cell scheme definitions for sensitivity analysis.'
    """,
    "ref.airspace_cells_v2": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.ref.airspace_cells_v2 (
        cell_id STRING NOT NULL,
        horizontal_cell_id STRING NOT NULL,
        vertical_cell_id STRING NOT NULL,
        horizontal_index_x INT NOT NULL,
        horizontal_index_y INT NOT NULL,
        vertical_index INT NOT NULL,
        min_latitude DOUBLE NOT NULL,
        max_latitude DOUBLE NOT NULL,
        min_longitude DOUBLE NOT NULL,
        max_longitude DOUBLE NOT NULL,
        center_latitude DOUBLE,
        center_longitude DOUBLE,
        min_altitude_ft DOUBLE NOT NULL,
        max_altitude_ft DOUBLE NOT NULL,
        horizontal_cell_nm DOUBLE NOT NULL,
        vertical_cell_ft DOUBLE NOT NULL,
        projection_mode STRING NOT NULL,
        focus_airport STRING NOT NULL,
        active_flag BOOLEAN NOT NULL,
        created_at TIMESTAMP NOT NULL,
        cell_scheme_id STRING NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Reference V2 3D airspace cells based on NM x ft discretization.'
    """,
    "brz_adsb.hist_state_vectors": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.brz_adsb.hist_state_vectors (
        time BIGINT NOT NULL,
        icao24 STRING NOT NULL,
        callsign STRING,
        lat DOUBLE,
        lon DOUBLE,
        velocity DOUBLE,
        heading DOUBLE,
        vertrate DOUBLE,
        onground BOOLEAN,
        alert BOOLEAN,
        spi BOOLEAN,
        squawk STRING,
        baroaltitude DOUBLE,
        geoaltitude DOUBLE,
        lastposupdate DOUBLE,
        lastcontact DOUBLE,
        hour BIGINT NOT NULL,
        source_query_start TIMESTAMP,
        source_query_end TIMESTAMP,
        ingested_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    PARTITIONED BY (hour)
    COMMENT 'Bronze historical state vectors extracted from OpenSky Trino for the Frankfurt scope.'
    """,
    "brz_adsb.hist_flights": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.brz_adsb.hist_flights (
        icao24 STRING NOT NULL,
        firstseen BIGINT NOT NULL,
        lastseen BIGINT NOT NULL,
        estdepartureairport STRING,
        estarrivalairport STRING,
        callsign STRING,
        day BIGINT NOT NULL,
        source_query_start TIMESTAMP,
        source_query_end TIMESTAMP,
        ingested_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    PARTITIONED BY (day)
    COMMENT 'Bronze historical flights extracted from OpenSky Trino for Frankfurt context analysis.'
    """,
    "brz_adsb.live_states": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.brz_adsb.live_states (
        snapshot_time TIMESTAMP NOT NULL,
        icao24 STRING NOT NULL,
        callsign STRING,
        origin_country STRING,
        latitude DOUBLE,
        longitude DOUBLE,
        baro_altitude_m DOUBLE,
        geo_altitude_m DOUBLE,
        velocity_mps DOUBLE,
        true_track_deg DOUBLE,
        vertical_rate_mps DOUBLE,
        on_ground BOOLEAN,
        scope_airport STRING NOT NULL,
        scope_radius_nm INT NOT NULL,
        ingested_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Bronze live state snapshots from the OpenSky REST API.'
    """,
    "slv_airspace.flight_states_clean": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.slv_airspace.flight_states_clean (
        event_time TIMESTAMP NOT NULL,
        time_window_5m TIMESTAMP NOT NULL,
        grid_id STRING NOT NULL,
        icao24 STRING NOT NULL,
        callsign STRING,
        latitude DOUBLE NOT NULL,
        longitude DOUBLE NOT NULL,
        altitude_ft DOUBLE,
        speed_kt DOUBLE,
        heading_deg DOUBLE,
        vertical_rate_fpm DOUBLE,
        altitude_band_id STRING,
        focus_airport STRING NOT NULL,
        source_table STRING NOT NULL,
        ingested_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Silver cleaned state vectors standardized for degree-grid complexity baselining.'
    """,
    "slv_airspace.flight_states_cellized_v2": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.slv_airspace.flight_states_cellized_v2 (
        event_time TIMESTAMP NOT NULL,
        analysis_window_start TIMESTAMP NOT NULL,
        horizontal_cell_id STRING NOT NULL,
        vertical_cell_id STRING NOT NULL,
        cell_id STRING NOT NULL,
        horizontal_index_x INT NOT NULL,
        horizontal_index_y INT NOT NULL,
        vertical_index INT NOT NULL,
        icao24 STRING NOT NULL,
        callsign STRING,
        latitude DOUBLE NOT NULL,
        longitude DOUBLE NOT NULL,
        altitude_ft DOUBLE,
        speed_kt DOUBLE,
        heading_deg DOUBLE,
        vertical_rate_fpm DOUBLE,
        focus_airport STRING NOT NULL,
        source_table STRING NOT NULL,
        cell_scheme_id STRING NOT NULL,
        ingested_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Silver V2 cellized state vectors using NM x ft discretization and configurable analysis windows.'
    """,
    "gld_airspace.grid_complexity_5m": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.grid_complexity_5m (
        window_start TIMESTAMP NOT NULL,
        grid_id STRING NOT NULL,
        aircraft_count BIGINT,
        heading_dispersion DOUBLE,
        altitude_dispersion DOUBLE,
        speed_dispersion DOUBLE,
        complexity_score DOUBLE,
        computed_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Gold 5-minute degree-grid complexity metrics for the Frankfurt baseline workflow.'
    """,
    "gld_airspace.complexity_hotspots": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.complexity_hotspots (
        grid_id STRING NOT NULL,
        avg_complexity_score DOUBLE,
        peak_complexity_score DOUBLE,
        num_high_complexity_windows BIGINT,
        ranking INT,
        computed_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Gold degree-grid hotspot ranking table for the Frankfurt baseline workflow.'
    """,
    "gld_airspace.complexity_trend_15m": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.complexity_trend_15m (
        window_start TIMESTAMP NOT NULL,
        avg_complexity_score DOUBLE,
        peak_complexity_score DOUBLE,
        active_grid_count BIGINT,
        active_aircraft_count BIGINT,
        computed_at TIMESTAMP NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Gold 15-minute trend table for the baseline Frankfurt workflow.'
    """,
    "gld_airspace.horizontal_complexity_v2": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.horizontal_complexity_v2 (
        window_start TIMESTAMP NOT NULL,
        horizontal_cell_id STRING NOT NULL,
        aircraft_count BIGINT,
        heading_dispersion DOUBLE,
        speed_dispersion DOUBLE,
        active_vertical_cells BIGINT,
        complexity_score DOUBLE,
        computed_at TIMESTAMP NOT NULL,
        cell_scheme_id STRING NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Gold V2 horizontal-cell complexity metrics for PRU-inspired terminal analysis.'
    """,
    "gld_airspace.horizontal_hotspots_v2": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.horizontal_hotspots_v2 (
        horizontal_cell_id STRING NOT NULL,
        active_windows BIGINT,
        avg_complexity_score DOUBLE,
        peak_complexity_score DOUBLE,
        num_high_complexity_windows BIGINT,
        ranking INT,
        computed_at TIMESTAMP NOT NULL,
        cell_scheme_id STRING NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Gold V2 hotspot ranking table with minimum active-window filtering.'
    """,
    "gld_airspace.complexity_trend_v2": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.complexity_trend_v2 (
        window_start TIMESTAMP NOT NULL,
        avg_complexity_score DOUBLE,
        peak_complexity_score DOUBLE,
        active_horizontal_cell_count BIGINT,
        active_aircraft_count BIGINT,
        computed_at TIMESTAMP NOT NULL,
        cell_scheme_id STRING NOT NULL,
        run_id STRING NOT NULL
    ) USING DELTA
    COMMENT 'Gold V2 trend table for sensitivity-ready complexity analysis.'
    """,
    "obs.ingestion_log": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.obs.ingestion_log (
        run_id STRING NOT NULL,
        pipeline_name STRING NOT NULL,
        source_name STRING NOT NULL,
        source_object STRING NOT NULL,
        target_table STRING NOT NULL,
        scope_id STRING NOT NULL,
        start_date DATE,
        end_date DATE,
        partition_key STRING,
        planned_partition_count INT,
        success_partition_count INT,
        failed_partition_count INT,
        dry_run BOOLEAN,
        status STRING NOT NULL,
        rows_written BIGINT,
        started_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP,
        error_message STRING,
        metadata_json STRING
    ) USING DELTA
    COMMENT 'Historical ingestion run log.'
    """,
    "obs.ingestion_partition_log": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.obs.ingestion_partition_log (
        run_id STRING NOT NULL,
        pipeline_name STRING NOT NULL,
        source_object STRING NOT NULL,
        target_table STRING NOT NULL,
        partition_key STRING NOT NULL,
        partition_value STRING NOT NULL,
        status STRING NOT NULL,
        rows_read BIGINT,
        rows_written BIGINT,
        attempt_no INT,
        dry_run BOOLEAN,
        started_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP,
        error_message STRING
    ) USING DELTA
    COMMENT 'Partition-level historical ingestion audit log.'
    """,
    "obs.live_snapshot_manifest": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.obs.live_snapshot_manifest (
        scope_id STRING NOT NULL,
        snapshot_time TIMESTAMP NOT NULL,
        snapshot_epoch BIGINT NOT NULL,
        source_object STRING NOT NULL,
        target_table STRING NOT NULL,
        status STRING NOT NULL,
        rows_read BIGINT,
        rows_written BIGINT,
        requested_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP,
        run_id STRING NOT NULL,
        error_message STRING
    ) USING DELTA
    COMMENT 'Manifest of live snapshot ingestion attempts and completion status.'
    """,
    "obs.pipeline_run_log": """
    CREATE TABLE IF NOT EXISTS `{catalog}`.obs.pipeline_run_log (
        run_id STRING NOT NULL,
        pipeline_name STRING NOT NULL,
        layer STRING NOT NULL,
        target_table STRING NOT NULL,
        status STRING NOT NULL,
        rows_read BIGINT,
        rows_written BIGINT,
        started_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP,
        error_message STRING,
        metadata_json STRING
    ) USING DELTA
    COMMENT 'Pipeline execution log for Bronze, Silver, Gold, and visualization workflows.'
    """,
}


In [ ]:
for schema_name in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.{schema_name}")

tables_ensured = []
for table_name, ddl in table_specs.items():
    spark.sql(ddl.format(catalog=catalog))
    tables_ensured.append(f"{catalog}.{table_name}")

def build_merge_sql(*, table_name: str, key_column: str, source_select_sql: str, source_columns: list[str]) -> str:
    target_columns = set(spark.table(table_name).columns)
    merge_columns = [column for column in source_columns if column in target_columns]
    update_assignments = ",\n            ".join(
        [f"target.{column} = source.{column}" for column in merge_columns]
    )
    insert_columns = ",\n            ".join(merge_columns)
    insert_values = ",\n            ".join([f"source.{column}" for column in merge_columns])
    return f"""
    MERGE INTO {table_name} AS target
    USING (
        {source_select_sql}
    ) AS source
    ON target.{key_column} = source.{key_column}
    WHEN MATCHED THEN UPDATE SET
            {update_assignments}
    WHEN NOT MATCHED THEN INSERT (
            {insert_columns}
    ) VALUES (
            {insert_values}
    )
    """

project_scope_source_sql = f"""
SELECT
    'fra_case_study_v1' AS scope_id,
    '{focus_airport}' AS focus_airport,
    {ingestion_radius_nm} AS ingestion_radius_nm,
    {bbox['min_latitude']} AS bbox_min_latitude,
    {bbox['max_latitude']} AS bbox_max_latitude,
    {bbox['min_longitude']} AS bbox_min_longitude,
    {bbox['max_longitude']} AS bbox_max_longitude,
    {complexity_window_minutes} AS complexity_window_minutes,
    {trend_window_minutes} AS trend_window_minutes,
    '{','.join(complexity_components)}' AS complexity_components,
    current_timestamp() AS created_at,
    'platform_setup' AS run_id
"""

spark.sql(
    build_merge_sql(
        table_name=f"`{catalog}`.ref.project_scope",
        key_column="scope_id",
        source_select_sql=project_scope_source_sql,
        source_columns=[
            "scope_id",
            "focus_airport",
            "ingestion_radius_nm",
            "bbox_min_latitude",
            "bbox_max_latitude",
            "bbox_min_longitude",
            "bbox_max_longitude",
            "complexity_window_minutes",
            "trend_window_minutes",
            "complexity_components",
            "created_at",
            "run_id",
        ],
    )
)

for sort_order, band in enumerate(altitude_bands, start=1):
    altitude_band_source_sql = f"""
    SELECT
        '{band['band_id']}' AS band_id,
        '{band['band_label']}' AS band_label,
        {sort_order} AS sort_order,
        {float(band['min_altitude_ft'])} AS min_altitude_ft,
        {float(band['max_altitude_ft'])} AS max_altitude_ft,
        current_timestamp() AS created_at,
        'platform_setup' AS run_id
    """
    spark.sql(
        build_merge_sql(
            table_name=f"`{catalog}`.ref.altitude_bands",
            key_column="band_id",
            source_select_sql=altitude_band_source_sql,
            source_columns=[
                "band_id",
                "band_label",
                "sort_order",
                "min_altitude_ft",
                "max_altitude_ft",
                "created_at",
                "run_id",
            ],
        )
    )

result = {
    'status': 'success',
    'catalog': catalog,
    'schemas_created': [f"{catalog}.{schema_name}" for schema_name in schemas],
    'table_count': len(table_specs),
    'tables_ensured': tables_ensured,
    'scope_defaults': {
        'scope_id': 'fra_case_study_v1',
        'focus_airport': focus_airport,
        'ingestion_radius_nm': ingestion_radius_nm,
        'bbox': bbox,
        'complexity_window_minutes': complexity_window_minutes,
        'trend_window_minutes': trend_window_minutes,
        'complexity_components': complexity_components,
    },
    'v2_defaults': v2_defaults,
}

result
